# Collaborative Filtering 
# Base of the filtering
The approach for the collaborative filtering is item-based. 
This results from the number of users with 1,103,602 being a lot bigger than the number of items (articles: 125,541). As well as that most of the users aren't subscribers. Therefore we don't have a lot of information about the users. The articles have more interactions than the users, the model should therefore be more stable than a user-based model. 
It needs to be noticed that the collaborative filtering with an item-based approach has the cold start problem with new articles.  


In [20]:
import pandas as pd
import math
from collections import defaultdict

Load the datasets

In [21]:
behaviors = pd.read_parquet("datas/ebnerd_demo/train/behaviors.parquet")  
history   = pd.read_parquet("datas/ebnerd_demo/train/history.parquet")    
articles  = pd.read_parquet("datas/ebnerd_demo/articles.parquet") 

Create the set of read articles from the users from the data sets behaviors and history

In [22]:
user_items = defaultdict(set)

# behaviors dataset: each row has one article_id
for _, row in behaviors.iterrows():
    u = row['user_id']
    art = row['article_id']
    if pd.notna(art):        user_items[u].add(str(art))

# history dataset: each row has article_id_fixed, an iterable
for _, row in history.iterrows():
    u = row['user_id']
    arts = row['article_id_fixed']
    # skip if not iterable or empty
    try:
        iterator = iter(arts)
    except TypeError:
        continue
    for art in arts:
        user_items[u].add(str(art))


Build the item frequencies and the co-occurrence counts per user

In [23]:
# number of all users
item_count = defaultdict(int)      
# number of the intersection of all users and articles
co_count   = defaultdict(lambda: defaultdict(int))  # |U_i ∩ U_j|

for u, arts in user_items.items():
    arts = list(arts)
    # Count each item's total users
    for art in arts:
        item_count[art] += 1
    # For every pair (i,j) read by this user, increment co-occurrence both ways
    for idx in range(len(arts)):
        for jdx in range(idx+1, len(arts)):
            i, j = arts[idx], arts[jdx]
            co_count[i][j] += 1
            co_count[j][i] += 1

Recommend a similar article for a given article while computing article-article similarity using Cosine Similarity

In [24]:
def recommend_similar_articles(article_id, top_n=5):
    """
    For the given article_id, looks up all co-read items j in co_count[article_id],
    computes sim(i,j) = co_count[i][j] / (sqrt(item_count[i]) * sqrt(item_count[j])),
    and returns the top_n neighbors.
    """
    i = str(article_id)
    if i not in co_count:
        return []

    sims = []
    norm_i = math.sqrt(item_count[i])
    for j, cij in co_count[i].items():
        norm_j = math.sqrt(item_count[j])
        sims.append((j, cij / (norm_i * norm_j)))

    # Sort descending by similarity score
    sims.sort(key=lambda x: x[1], reverse=True)
    return [j for j, score in sims[:top_n]]


For tests: 

In [25]:
some_art = next(iter(co_count), None)
if some_art:
    print(f"Top 5 articles similar to {some_art}:")
    for nbr in recommend_similar_articles(some_art, 5):
        print(" ", nbr)
else:
    print("No co‑occurrence data—check your inputs/paths.")
    


Top 5 articles similar to 9779538.0:
  9779289.0
  9769557.0
  9779520.0
  9518125
  9754160.0
